<h1><center>Laboratorio 7: Ensamblaje, Optimización de Hiperparámetros e Interpretabilidad 🤖</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco y Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Bryan Cabezas
- Nombre de alumno 2: Gonzalo Sobarzo

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Antigravity, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

## Temas a tratar

- Ensamblaje: Bagging (`RandomForest`), Boosting (`XGBoost`, `LightGBM`) y Stacking.
- Optimización de Hiperparámetros con `Optuna` y visualización interactiva con `optuna-dashboard`.
- Interpretabilidad global: `Permutation Feature Importance (PFI)`.
- Interpretabilidad local: `SHAP`.

### Objetivos principales del laboratorio

- Aplicar y comparar métodos de ensamblaje sobre un problema de clasificación de texto.
- Optimizar hiperparámetros de LightGBM usando Optuna y visualizar el proceso con `optuna-dashboard`.
- Interpretar las predicciones del modelo usando PFI y SHAP.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de Python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`.

### Instalamos librerías 😸

In [9]:
!uv add nltk lightgbm xgboost optuna shap scikit-learn plotly certifi

Resolved 144 packages in 665ms                                       
⠙ slicer==0.0.8                                                                 Audited 136 packages in 23ms


In [2]:
import ssl
import certifi

ssl._create_default_https_context = (
    lambda: ssl.create_default_context(cafile=certifi.where())
) #ESTO FUE PARA DESCARGAR STOPWORDS DE NLTK EN MAC

import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/bryan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
import warnings

import matplotlib.pyplot as plt
import nltk
import numpy as np
import optuna
import pandas as pd
import shap
from lightgbm import LGBMClassifier
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split  # noqa: F401
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from optuna.visualization import (
    plot_parallel_coordinate,
    plot_optimization_history,
    plot_param_importances,
)
import plotly.express as px
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

RANDOM_STATE = 42
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/bryan/Desktop/MDS/segundo semestre/Laboratorio de ciencia de datos/Repo/MDS7202/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. ¿Quién es Bat Cow?

<p align="center">
  <img src="https://i.imgur.com/D9f1RHy.jpg" width="350">
</p>

En vez de estar desarrollando las evaluaciones correspondientes a su curso, su profesor de catedra y su auxiliar discuten acerca la alineación (i.e., si es heroe o villano) del personaje de ficción Bat-Cow.

El cuerpo docente, no logra ponerse de acuerdo si el personaje es bueno, neutral o malo: el auxiliar plantea que Bat-cow posee una siniestra mirada, intrigante pero común característica de los personajes malvados.
Por otra parte, extendiendo las ideas de Rousseau, el profesor plantea que tal como los humanos no nacen malos, no existe motivo por el cual una vaca con superpoderes deba serlo.

Sin embargo, ambos concuerdan que es difícil estimar la alineación solo usando los atributos físicos. Es por esto que les solicitan construir y optimizar un clasificador basado en texto que analice la alineación de cada personaje basado en su historia personal.

Para este laboratorio deben trabajar con los datos `df_comics.csv` y `comics_no_label.csv` subidos a u-cursos.

In [4]:
df_comics = pd.read_csv("df_comics.csv", index_col=0)
df_comics_no_label = pd.read_csv("comics_no_label.csv", index_col=0)
df_comics = df_comics.dropna(subset=["history_text"])
df_comics

,name,real_name,full_name,overall_score,history_text,powers_text,intelligence_score,strength_score,speed_score,durability_score,...,has_flight,has_accelerated_healing,has_weapons_master,has_intelligence,has_reflexes,has_super_speed,has_durability,has_stamina,has_agility,has_super_strength
0,3-D Man,"Delroy Garrett, Jr.","Delroy Garrett, Jr.",6,"Delroy Garrett, Jr. grew up to become a track ...",NaN,85,30,60,60,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,A-Bomb,Richard Milhouse Jones,Richard Milhouse Jones,20,"Richard ""Rick"" Jones was orphaned at a young ...","On rare occasions, and through unusual circu...",80,100,80,100,...,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0
3,Aa,Aa,NaN,12,Aa is one of the more passive members of the P...,NaN,80,50,55,45,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Aaron Cash,Aaron Cash,Aaron Cash,5,Aaron Cash is the head of security at Arkham A...,NaN,80,10,25,40,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,Aayla Secura,Aayla Secura,NaN,8,ayla Secura was a Rutian Twi'lek Jedi Knight (...,NaN,90,40,45,55,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1445,Zatanna,Zatanna Zatara,Zatanna Zatara,10,Zatanna is the daughter of adventurer John Zat...,Zatanna is genetically talented with her magi...,90,10,25,30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1446,Zero,DWN-∞: Zero,DWN-∞: Zero,18,Zero was created by the late Dr. Albert Wily ...,NaN,80,100,100,100,...,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1447,Zoom (New 52),Hunter Zolomon,NaN,20,"Hunter Zolomon is better known as Zoom, a spee...",After tricking Barry Allen and Wally West into...,95,50,100,75,...,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1448,Zoom,Hunter Zolomon,Hunter Zolomon,9,Hunter Zolomon had a troubled relationship wi...,"Zoom is able to alter time, to make himself ev...",75,10,100,30,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## 1.1 Obtención de Features y Bag of Words

<p align="center">
  <img src="https://media0.giphy.com/media/eIUpSyzwGp0YhAMTKr/200.gif" width="300">
</p>

`bag of words` es un modelo de conteo utilizado en NLP que genera una representación vectorial para cada documento a través del conteo de las palabras que contienen.

<p align="center">
  <img src="https://user.oc-static.com/upload/2020/10/23/16034397439042_surfin%20bird%20bow.png" width="500">
</p>

Para facilitar el conteo transformamos cada documento en un vector mediante **tokenización**:

In [13]:
docs = ["The teacher rocks like a good rock & roll", "the rock is the best actor in the world"]
docs_tokenizados = [word_tokenize(doc) for doc in docs]
docs_tokenizados

[['The', 'teacher', 'rocks', 'like', 'a', 'good', 'rock', '&', 'roll'],
 ['the', 'rock', 'is', 'the', 'best', 'actor', 'in', 'the', 'world']]

Podemos mejorar la tokenización con:

- **Stemming**: transforma palabras a su forma raíz (*running → run*, *rocks → rock*).
- **Eliminación de Stopwords**: elimina palabras muy frecuentes que entorpecen la clasificación (*the*, *is*, *a*, ...).

<p align="center">
  <img src="https://devopedia.org/images/article/218/8583.1569386710.png" width="300">
</p>

In [6]:
stop_words = stopwords.words("english")


class StemmerTokenizer:
    def __init__(self):
        self.ps = PorterStemmer()

    def __call__(self, doc):
        doc_tok = word_tokenize(doc)
        doc_tok = [t for t in doc_tok if t not in stop_words]
        return [self.ps.stem(t) for t in doc_tok]


tokenizador = StemmerTokenizer()

docs = [
    "The teacher rocks like a good rock & roll",
    "the rock is the best actor in the world",
    "New York is a beautiful city",
]

print("Con StemmerTokenizer:")
print([tokenizador(doc) for doc in docs])
print("\nSin preprocesamiento:")
print([word_tokenize(doc) for doc in docs])

Con StemmerTokenizer:
[['the', 'teacher', 'rock', 'like', 'good', 'rock', '&', 'roll'], ['rock', 'best', 'actor', 'world'], ['new', 'york', 'beauti', 'citi']]

Sin preprocesamiento:
[['The', 'teacher', 'rocks', 'like', 'a', 'good', 'rock', '&', 'roll'], ['the', 'rock', 'is', 'the', 'best', 'actor', 'in', 'the', 'world'], ['New', 'York', 'is', 'a', 'beautiful', 'city']]


#### Al Estilo Scikit

Scikit implementa `bag of words` con `CountVectorizer()`. Además soporta **n-gramas**: secuencias contiguas de n palabras que se tratan como un único token. Esto permite capturar contexto local que los unigramas pierden.

| Tipo | n | Tokens de `"nueva york ciudad"` |
|------|---|--------------------------------|
| Unigrama | 1 | `nueva`, `york`, `ciudad` |
| Bigrama | 2 | `nueva york`, `york ciudad` |
| Trigrama | 3 | `nueva york ciudad` |

Con `ngram_range=(1,2)` el vectorizador incluye **unigramas y bigramas** simultáneamente. Los bigramas son especialmente útiles para capturar expresiones compuestas como `bat cow`, `spider man` o `super hero` que pierden su significado si se separan.

El parámetro `max_features` limita el vocabulario a los n tokens más frecuentes, controlando la dimensionalidad de la representación.

In [15]:
bow = CountVectorizer(tokenizer=StemmerTokenizer(), ngram_range=(1, 2))
df_bow = bow.fit_transform(docs)
pd.DataFrame(df_bow.toarray(), columns=bow.get_feature_names_out())

,&,& roll,actor,actor world,beauti,beauti citi,best,best actor,citi,good,...,rock,rock &,rock best,rock like,roll,teacher,teacher rock,world,york,york beauti
0,1,1,0,0,0,0,0,0,0,1,...,2,1,0,1,1,1,1,0,0,0
1,0,0,1,1,0,0,1,1,0,0,...,1,0,1,0,0,0,0,1,0,0
2,0,0,0,0,1,1,0,0,1,0,...,0,0,0,0,0,0,0,0,1,1


#### Combinando Features: `ColumnTransformer`

Para combinar en un solo paso el preprocesamiento de texto y numérico, usamos `ColumnTransformer`. Este aplica transformadores distintos a subconjuntos de columnas del DataFrame y concatena el resultado en una sola matriz de features lista para entrenar.

<p align="center">
  <img src="https://c.tenor.com/LkQzw7k5DV4AAAAd/anime-hacking.gif" width="300">
</p>

El `preprocessing_transformer` que usaremos a lo largo del lab combina:

- **`CountVectorizer`** con `StemmerTokenizer`, `ngram_range=(1,2)` y `max_features=500` → aplicado sobre la columna `history_text`.
- **`MinMaxScaler`** → aplicado sobre los 6 atributos numéricos de habilidad: `intelligence_score`, `strength_score`, `speed_score`, `durability_score`, `power_score`, `combat_score`.

In [7]:
preprocessing_transformer = ColumnTransformer(
    transformers=[
        (
            "MinMaxScaler",
            MinMaxScaler(),
            [
                "intelligence_score",
                "strength_score",
                "speed_score",
                "durability_score",
                "power_score",
                "combat_score",
            ],
        ),
        (
            "bow",
            CountVectorizer(
                tokenizer=StemmerTokenizer(),
                max_features=500,
                ngram_range=(1, 2),
            ),
            "history_text",
        ),
    ]
)

## 1.2 Diseño de Baseline y Primer Entrenamiento [1 Punto]

<p align="center">
  <img src="https://pa1.narvii.com/6374/9eaec1b7bf9157334151452a669516f9a78b954c_hq.gif" width="300">
</p>

### 1.2.1 ¿Qué es un Baseline? [0.2 Puntos]

Antes de entrenar modelos complejos, es fundamental establecer un punto de referencia mínimo. Responde las siguientes preguntas con tus propias palabras:

1. **¿Qué es un baseline en Machine Learning?** ¿Para qué sirve establecerlo antes de evaluar modelos más sofisticados?
2. **¿Por qué usamos un `DummyClassifier` como baseline?** ¿Qué implica que un modelo "real" no logre superar su rendimiento?

> **Respuesta:**

# Escribe aquí la respuesta
- 1. El baseline es un modelo de clasificación que se utiliza como referencia y como base para evaluar otros modelos más sotisficados. Se establece para comparar las métricas de rendimiento contra los otros modelos y ver si los modelos más complejos realmente aportan una mejora.
- 2. Se utiliza DummyClassifier como baseline ya que es un modelo simple que hace predicciones basadas en la distribución de las clases de los datos, sin aprender de los datos de entrada y de manera aleatoria. Si un modelo más complejo no supera el rendimiento, significa que no está aprendiendo un patron útil de los datos y no es mejor que un modelo basado en aleatorización, por lo cúal es mejor descartarlo.

---

### 1.2.2 Implementación [0.6 Puntos]

Genere un `Pipeline` con las características de 1.1 y un `DecisionTreeClassifier()` por defecto.

Separe el dataset en entrenamiento/prueba (80/20, estratificado, `random_state=RANDOM_STATE`). Entrene, reporte `classification_report` y compare con un `DummyClassifier(strategy="stratified")`.

**To-do:**
- [ ] Pipeline con preprocesamiento → `DecisionTreeClassifier`.
- [ ] Holdout estratificado 80/20.
- [ ] `classification_report` del baseline.
- [ ] Entrenar `DummyClassifier` y comparar.

In [8]:
#### Código aquí ####
pipeline_dt = Pipeline(
    steps=[
        ("preprocesamiento", preprocessing_transformer),
        ("modelo", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    df_comics.drop(columns=["alignment"]), df_comics["alignment"], test_size=0.2, random_state=RANDOM_STATE, stratify=df_comics["alignment"])

pipeline_dt.fit(X_train, y_train)
y_pred_dt = pipeline_dt.predict(X_test)

print("Classification Report - Decision Tree:")
print(classification_report(y_test, y_pred_dt))

Classification Report - Decision Tree:
              precision    recall  f1-score   support

         Bad       0.47      0.48      0.47        86
        Good       0.68      0.64      0.66       148
     Neutral       0.10      0.13      0.12        23

    accuracy                           0.54       257
   macro avg       0.42      0.42      0.42       257
weighted avg       0.56      0.54      0.55       257



In [9]:
pipeline_dummy = Pipeline(
    steps=[
        ("preprocesamiento", preprocessing_transformer),
        ("modelo", DummyClassifier(strategy="stratified", random_state=RANDOM_STATE))
    ]
)

pipeline_dummy.fit(X_train, y_train)
y_pred_dummy = pipeline_dummy.predict(X_test)

print("Classification Report - Dummy Classifier:")
print(classification_report(y_test, y_pred_dummy))

Classification Report - Dummy Classifier:
              precision    recall  f1-score   support

         Bad       0.32      0.34      0.33        86
        Good       0.58      0.56      0.57       148
     Neutral       0.12      0.13      0.12        23

    accuracy                           0.45       257
   macro avg       0.34      0.34      0.34       257
weighted avg       0.46      0.45      0.45       257



### 1.2.3 Pregunta de Cierre [0.2 Puntos]

**Pregunta:** ¿El `DecisionTreeClassifier` supera al `DummyClassifier`? ¿Qué concluyes de esto sobre lo que ha aprendido el modelo? Además responde:

1. ¿Por qué el accuracy puede ser una métrica engañosa en este problema? ¿Qué métrica es más apropiada si las clases están desbalanceadas?
2. ¿Por qué se usa el parámetro `stratify` en el `train_test_split`? ¿Qué problema evitamos al usarlo?
3. ¿Es mejor el clasificador que su versión aleatoria? ¿Podemos avanzar con confianza de que estamos clasificando mejor que si por ejemplo, tiraramos un dado con 3 caras?

> **Respuesta:**

In [25]:
df_comics['alignment'].value_counts()

alignment
Good       743
Bad        429
Neutral    113
Name: count, dtype: int64

# Escribe aquí la respuesta
El modelo DecisionTreeClassifier efectivamente supera al DummyClassifier en las métricas globales, alcanzando un Accurary de 54% frente al 45% de la línea base, y un Macro F1-Score de 0.42 frente a 0.34. Esto demuestra que el árbol no esta prediciendo al azar y que logró encontrar patrones a partir de las características de los datos para diferenciar las clases mayoritarias. Aún asi, el modelo presenta un rendimiento muy bajo en la clase minoritaria "Neutral" de 0.12 en F1-score. Como también el rendimiento aunque mejoró, sigue presentando un modelo con un rendimiento pobre, confirmando que sigue siendo un modelo no confiable.

- 1. Al presentar un desbalanceo de datos, con 743 buenos, 429 malos y 113 neutrales, el modelo aprendera de mejor manera los patrones de la clase mayoritaria al presentar más datos, por lo cual el modelo se equivocará en menor medida en esos datos, haciendo que el accuracy aumente, pero puede ocurrir que no clasifique correctamente las clases minoritarias, las cuales en su mayoría son las más importantes al presentar en menor medida, por lo cual esa métrica estaría sesgada. Una métrica más apropiada sería F1-Score, ya que calcula la media armónica entre la Precisión (cuántos de los clasificados como positivos eran realmente positivos) y el Recall (cuántos de los positivos reales logró capturar el modelo). Al balancear ambas, el F1-score penaliza los modelos que predicen únicamente la clase más frecuente, exigiendo un buen desempeño tanto en la clasificación de la clase mayoritaria como en la minoritaria.

- 2. Se utiliza stratify en la división de entrenamiento y testeo, debido que mantiene la misma propoción en ambos conjuntos, evitando el problema de que en un conjunto únicamente existan datos de la clase mayoritaria y cero o muy pocos valores de la clase minoritaria, generando un modelo que no generalice correctamente.


- 3. Sí, el clasificador Decision Tree es mejor que su versión aleatoria y que un dado de 3 caras, pero la mejora no es sustancial por lo que no se puede avanzar con confianza. Un dado perfecto de 3 caras acertaría el 33.3% de las veces, el árbol de decisión logra un 54% de Accuracy, lo cual es más alto. Sin embargo, el Macro F1-Score (0.42 del árbol vs 0.34 del Dummy) muestra un rendimiento real en la clasificación de las categorías muy pobre. EL modelo tiene un desempeño muy bajo en la clase "Neutral" de 0.12 en F1-Score, siendo practicamente similar al azar. Por lo que en si, no es un modelo confiable en producción, superando apenas la línea base del comportamiento aleatorio. 

---

# 2. Métodos de Ensamblaje [2 Puntos]

<p align="center">
  <img src="https://media.giphy.com/media/l0HlHFRbmaZtBRhXG/giphy.gif" width="300">
</p>

Los métodos de ensamblaje combinan múltiples modelos para obtener predicciones más robustas. Exploraremos tres estrategias:

| Estrategia | Idea clave | Ejemplo |
|------------|-----------|---------|
| **Bagging** | Modelos en paralelo sobre subconjuntos aleatorios | Random Forest |
| **Boosting** | Modelos en secuencia, cada uno corrige al anterior | XGBoost, LightGBM |
| **Stacking** | Predicciones de modelos base como input de un meta-modelo | StackingClassifier |

Todos usarán el mismo `preprocessing_transformer` de la sección 1.

## 2.1 Bagging: Random Forest [0.5 Puntos]

### 2.1.1 Descripción del algoritmo [0.2 Puntos]

Describe con tus propias palabras cómo funciona el **Bagging (Bootstrap Aggregating)**. La descripción debe cubrir los siguientes tres pasos:

1. **Generación de subconjuntos**: ¿Cómo se obtienen los subconjuntos de entrenamiento a partir del dataset original? ¿Se usa todo el dataset en cada uno? ¿Se pueden repetir instancias?
2. **Entrenamiento**: ¿Qué se entrena sobre cada subconjunto? ¿Los modelos se entrenan de forma dependiente o independiente entre sí?
3. **Agregación**: ¿Cómo se combinan las predicciones de todos los modelos para obtener una respuesta final?

> **Respuesta:**


# Escribe aquí la respuesta
- El bagging es un método de ensamble diseñada para mejorar la estabilidad y precisión de los modelo, reduciendo la varianza y evitando sobreajuste.
- Primero genera múltiples subconjuntos de datos a partir del dataset original, mediante el método llamado muestreo con reemplazo, donde aleatoriamente se obtiene un dato y se registra en un sobconjunto y nuevamente ese registro se agrega al conjunto inicial al ser con reemplazo. No se utiliza todo el dataset en los subconjuntos. Cada subconjunto tiene el mismo tamaño (N) que el dataset original, pero debido al azar no contiene los mismos datos originales, sino que algunos repetidos y otros no. 
- Una vez creados los M subconjuntos obtenidos en el paso anterior, comienza la etapa de Entrenamiento, donde sobre cada subconjunto se entrena un modelo base individual, utilizando árboles de decisión complejos profundos sin podar, con alta varianza y bajo sesgo, donde cada arbol se entrena exclusivamente con su subconjunto correspondiente. Este entrenamiento de modelos se realiza completamente independiente entre sí, es decir, un arbol no necesita saber lo que aprende otro, ya que entrenan de forma aislada con su respectivo subconjunto, pudiendo encontrar patrones que otros modelos no lo obtuvieron. Normalmente, se busca que los arboles no esten correlacionados, pero con bagging solo, esto no se garantiza, ya que en caso de existir variales dominantes probablemente esten en cada subconjunto, por lo que se puede agregar muestreo de columna para solucionar esto (como randomForest)
- Finalmente, las predicciones de todos los modelos se combina de dos maneras:
    - En caso de ser un problema de clasificación, se utiliza la mayoría absoluta, es decir votación, donde se cuenta cuántos modelos predijeron cada clase y la clase que reciba más votos se convierte en la predicción final del conjunto.
    - Si es un problema de regresión, se calcula el promedio matemático simple de todas las predicciones entregadas por los modelos individuales.

Mediante la votación o promedio hace que modelos independientes que sufren alta varianza, los errores individuales se canelan entre sí, genenrando un modelo robusto que mantiene un sesgo bajo pero con una varianza controlada. 

---

### 2.1.2 Implementación [0.2 Puntos]

**To-do:**
- [ ] Pipeline con `RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)`.
- [ ] Entrenar y reportar `classification_report`.

In [10]:
#### Código aquí ####
pipeline_rf = Pipeline(
    steps=[
        ("preprocesamiento", preprocessing_transformer),
        ("modelo", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
    ]
)

pipeline_rf.fit(X_train, y_train)
y_pred_rf = pipeline_rf.predict(X_test)

print("Classification Report - Random Forest:")
print(classification_report(y_test, y_pred_rf))

Classification Report - Random Forest:
              precision    recall  f1-score   support

         Bad       0.67      0.35      0.46        86
        Good       0.66      0.93      0.77       148
     Neutral       0.00      0.00      0.00        23

    accuracy                           0.65       257
   macro avg       0.44      0.43      0.41       257
weighted avg       0.60      0.65      0.60       257



### 2.1.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** ¿El Random Forest mejoró respecto al baseline? Comenta los resultados observados en el `classification_report` y explica a qué se debe la diferencia (o falta de ella), considerando las características del algoritmo que describiste anteriormente.

> **Respuesta:**
    Random Forest mejoró el rendimiento global en comparación al baseline, elevando el Accuracy del 45% a 65% respectivamente y el Macro F1-Score de 0.34 a 0.41. Sin embargo, el rendimiento obtenido no presenta una gran mejora en general, debido a que el modelo busca maximizar la pureza global de los nodos (minimizando la entropia), por lo cual encuentra que la manera más eficiente de reducir el error es ignorando la clase minoritaria y especializándose en la clase mayoritaria; por esto mismo el modelo obtuvo 0.00 en todas las métricas de la clase "Neutral". Tomando en cuenta la descripción anterior de las etapas del modelo, en el primer paso del Bootstrapping, los subconjuntos creados presentaron en su mayoría datos de la clase mayoritaria "Good", generando que en el entrenamiento independiente los modelos aprendieran principalmente los patrones de dicha clase, alcanzando un recall de 0.93; en contraste, al presentar una baja cantidad de la clase minoritaria "Neutral", el modelo decidio ignorar las ramas que utilizan esos datos al no aportar significativamente al beneficio matemático global. Finalmente, en la etapa de agregación por votación, se consolida un sesgo hacia la clase "Bad", ya que logra una precision de 0.67 pero un recall de 0.36, demostrando que el modelo presenta una tendencia a aplastar los votos de los pocos árboles que detectador la clase correcta en favor de la clase "Good" debido a su alta mayoría.

    Es decir, el modelo Random Forest mejora el rendimiento general gracias a la especialización al grupo mayoritario, pero no logra resolver una clasificación correcta hacia las clases minoritarias, comportandose de forma muy similar a un clasificador dummy sofisticado. Una solución seria aplicar la configuración de class_weight en el modelo.

## 2.2 Boosting: XGBoost y LightGBM [0.8 Puntos]

### 2.2.1 Descripción del algoritmo [0.3 Puntos]

Describe con tus propias palabras cómo funciona el **Boosting**. Tu descripción debe cubrir los siguientes tres pasos:

1. **Entrenamiento secuencial**: ¿En qué se diferencia el Boosting del Bagging en cuanto al orden en que se entrenan los modelos? ¿Son independientes entre sí?
2. **Corrección de errores**: ¿Cómo sabe cada modelo nuevo en qué instancias debe enfocarse? ¿Qué información del modelo anterior utiliza?
3. **Predicción final**: ¿Cómo se combinan las predicciones de todos los modelos? ¿Es una votación simple o una combinación ponderada?

Además, explica brevemente en qué se diferencian **XGBoost** y **LightGBM** como implementaciones de Boosting, y por qué XGBoost requiere que las etiquetas sean numéricas mientras que LightGBM acepta strings directamente.

> **Respuesta:**

# Escribe aquí la respuesta

El boosting es un método de ensamble que trabaja de manera opuesta que bagging, en lugar de construir varios modelos, busca crear un modelo fuerte de manera incremental, el cual aprende de los errores cometidos por los modelos anteriores de manera secuencial y los intenta corregir en el siguiente modelo. Su objetivo es reducir el sesgo, utilizando modelos simples y llevarlo a un clasificador robusto.

- En el entrenamiento, Bagging genera árboles que se entrenan de forma paralela y aislada con diferentes subconjuntos de datos, de lo contrario, Boosting, los modelos se entrenan uno detras de otro, en fila, es decir, primero se entrena un modelo, luego el siguiente y así sucesivamente. Además, en este caso los modelos son dependientes, ya que el siguiente modelo debe entrenarse despues de que finalice el modelo anterior y obtenido sus resultados, ya que el siguiente modelo necesita conocer los datos clasificados erroneamente para corregirlos.

- El modelo nuevo sabe en que instancia enfocarse de los errores gracias a la retroalimentación de errores del modelo anterior, lo cual depende del algoritmo de Boosting específico. Por un lado, AdaBoost, le asigna mayor peso a los datos que fueron predichos erroneamente por el modelo anterior y menor peso a los datos correctamente clasificados, por lo cual el siguiente modelo recibira el dataset modificado y se enfocará en aprender patrones de los errores para poder corregirlos. Por otro lado, Gradient Boosting, en vez de utilizar pesos, lo que realiza es que el siguiente modelo se entrena directamente y únicamente con los errores (residuos) del modelo anterior, de forma que pueda corregir esa brecha.

- Boosting realiza una combinación ponderada de modelos, de manera que se le asigna un mayor peso de confianza a los modelos más precisos, es decir que tuvieron menos errores. Posteriormente para obtener la predicción final, se suman las predicciones de todos los modelos multiplicado por sus respectivos pesos de confianza asignados, generando un modelo más robusto.

La diferencia entre XGBoost y LightGBM es que XGBoost presenta una restricción estricta por niveles, donde esta obligado a evaluar la ganancia de información y dividir todos los nodos de un mismo nivel antes de permitir que cualquiera de ellos se extienda a una nueva profundidad(nivel) del árbol. Si bien no divide los nodos que no aportan ganancia (gracias a un umbral minimo o criterios de poda), su estructura se expande de forma horizontal. Como resultado generará un árbol balanceado, ordenado y simétrico. Aunque controla el sobreajuste, puede llegar a tomar más tiempo y cómputo. De lo contrario, LightGBM, elimina la noción de niveles equilibrados del árbol, sino que busca maximizar la ganancia de información, de manera que en cada nivel evalua la ganancia de información que ofrece dividir cada hoja y la que presenta mayor ganancia sigue por dicho camino y la divide, dejando las demás hojas congeladas. Esto provoca que LightGBM genere menos nodos totales, creando árboles muy asimétricos, donde una sola rama puede generar una alta profundidad. Asimismo, consume menos memoria y es más rápido, aunque requiere mayor cuidado con datasets pequeños para evitar sobreajuste.

Respecto al manejo de datos, XGboost requiere etiquetas numéricas debido que fue construido bajo una arquitectura matemática que procesa matrices y no tiene un preprocesamiento interno capaz de transformar texto. En cambio, LightGBM presenta un procesamiento interno que transforma las variables de texto a valores numéricos basados en histogramas de frecuencia que mapean cada categorías según su relación el target antes de entrenar los árboles.

---

### 2.2.2 Implementación [0.4 Puntos]

**To-do:**
- [ ] Crear `LabelEncoder`, ajustarlo sobre `y_train` y transformar `y_train` e `y_test`.
- [ ] Pipeline con `XGBClassifier(random_state=RANDOM_STATE, eval_metric="mlogloss", verbosity=0)`. Reportar resultados decodificando las predicciones con `le.inverse_transform`.
- [ ] Pipeline con `LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)`. Reportar resultados.

In [12]:
#### Código aquí ####

LabelEncoder_alignment = LabelEncoder().fit(y_train)
y_train_encoded = LabelEncoder_alignment.transform(y_train)
y_test_encoded = LabelEncoder_alignment.transform(y_test)

In [13]:
pipeline_XGBoost = Pipeline(
    steps=[
        ("preprocesamiento", preprocessing_transformer),
        ("modelo", XGBClassifier(random_state=RANDOM_STATE, eval_metric='mlogloss', verbosity=0))
    ]
)

pipeline_XGBoost.fit(X_train, y_train_encoded)
y_pred_XGBoost_encoded = pipeline_XGBoost.predict(X_test)
y_pred_XGBoost = LabelEncoder_alignment.inverse_transform(y_pred_XGBoost_encoded)

print("Classification Report - XGBoost:")
print(classification_report(y_test, y_pred_XGBoost))

Classification Report - XGBoost:
              precision    recall  f1-score   support

         Bad       0.56      0.42      0.48        86
        Good       0.68      0.84      0.75       148
     Neutral       0.50      0.17      0.26        23

    accuracy                           0.64       257
   macro avg       0.58      0.48      0.50       257
weighted avg       0.62      0.64      0.62       257



In [19]:
pipeline_LGBMC = Pipeline(
    steps=[
        ("preprocesamiento", preprocessing_transformer),
        ("modelo", LGBMClassifier(random_state=RANDOM_STATE, verbose=-1))
    ]
)
pipeline_LGBMC.fit(X_train, y_train)
y_pred_LGBMC = pipeline_LGBMC.predict(X_test)

print("Classification Report - LightGBM:")
print(classification_report(y_test, y_pred_LGBMC))

Classification Report - LightGBM:
              precision    recall  f1-score   support

         Bad       0.60      0.56      0.58        86
        Good       0.72      0.86      0.78       148
     Neutral       0.00      0.00      0.00        23

    accuracy                           0.68       257
   macro avg       0.44      0.47      0.45       257
weighted avg       0.61      0.68      0.64       257



### 2.2.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** Compara los resultados de XGBoost y LightGBM según el `classification_report`. ¿Cuál tuvo mejor desempeño en F1-Macro? ¿Ambos mejoran respecto al baseline? Considerando las diferencias que describiste en 2.2.1, ¿a qué atribuyes las similitudes o diferencias en rendimiento?

> **Respuesta:**

# Escribe aquí la respuesta

Tanto XGBoost como LightGBM lograron superar el modelo baseline, elevando el Macro F1-Score de 0.34 a valores de 0.50 y 0.45 respectivamente. Al comparar ambas implementaciones de Boosting, XGboost obtuvo un mejor desempeño en el F1-Macro con 0.50 frente a 0.45 de LightGBM. A pesar de que LightGBM consiguió un mayor valor de Accuracy gracias a enfocarse en las clases mayoritarias, fracasó en la categoría minoritaria "Neutral" obteniendo un valor de 0.00 en todas las métricas de dicha clase, repitiendo el mismo problema que Random Forest. En contraste, XGBoost fue capaz de capturar patrones de la clase minoritaria, logrando un Precision de 0.50, un Recall de 0.17 y un F1-Score de 0.26, lo que explica por qué su promedio ponderado por clase de F1-Macro fue superior en comparación a los dos modelos.

Las diferencias y similitudes entre modelos se basa en la estrategia de crecimiento de árboles. XGboost, al verse obligado en evaluar de manera horizontal y simétrica todos los nodos del árbol, permitió generar ramas equilibradas las cuales fueron capaces de aislar los patrones sutiles de los 23 registros de la clase minoritaria Neutral. Por el contrario, LightGBM al utilizar la estrategia por hojas que busca maximizar la ganancia, encontró más rentable dividir la masa de datos en las clases mayoritarias al aportar mayor ganancia de información y seguir dicho camino, ignorando la rama minoritaria al no aportar reducción significativa del gradiente global. Por esto mismo, XGBoost demostró una arquitectura más robusta para enfrentar el desbalanceo de datos.

## 2.3 Stacking [0.7 Puntos]

### 2.3.1 Descripción del algoritmo [0.2 Puntos]

Describe con tus propias palabras cómo funciona el **Stacking**. Tu descripción debe cubrir los siguientes tres aspectos:

1. **Predicciones como features**: ¿Qué rol cumplen los modelos base? ¿Sobre qué datos generan sus predicciones para ser usadas por el meta-modelo? ¿Por qué se usa validación cruzada interna en lugar de predecir directamente sobre los datos de entrenamiento?
2. **Meta-modelo**: ¿Qué recibe como input el meta-modelo y qué aprende? ¿En qué se diferencia su rol del de los modelos base?
3. **Ventaja sobre selección simple**: ¿Por qué el Stacking puede superar a cualquier modelo base individual? ¿Qué aprovecha de la diversidad entre modelos?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

### 2.3.2 Implementación [0.4 Puntos]

**Restricciones:**
- Mínimo **3 modelos base distintos**.
- Solo clasificadores básicos de scikit-learn: `LogisticRegression`, `MultinomialNB`, `SGDClassifier`, `DecisionTreeClassifier`, etc. **No se permiten modelos de ensamblaje** (`RandomForest`, `XGBoost`, `LightGBM`).
- El meta-modelo es de libre elección. Justifica tu elección.

**To-do:**
- [ ] Definir al menos 3 modelos base (solo clasificadores básicos de scikit-learn).
- [ ] Elegir un meta-modelo.
- [ ] Pipeline con `StackingClassifier(cv=3, n_jobs=-1)`.
- [ ] Reportar `classification_report`.

In [16]:
#### Código aquí ####
lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
mn = MultinomialNB()

estimadores = [
    ("Logistic Regression", lr),
    ("Decision Tree", dt),
    ("Multinomial NB", mn)
]

pipeline_stacking = Pipeline(
    steps=[
        ("preprocesamiento", preprocessing_transformer),
        ("modelo", StackingClassifier(estimators=estimadores,
                                      final_estimator=LogisticRegression(random_state=RANDOM_STATE),
                                      cv = 3,
                                      n_jobs=-1))
    ]
)

pipeline_stacking.fit(X_train, y_train)
y_pred_stacking = pipeline_stacking.predict(X_test)

print("Classification Report - Stacking Classifier:")
print(classification_report(y_test, y_pred_stacking))


Classification Report - Stacking Classifier:
              precision    recall  f1-score   support

         Bad       0.57      0.55      0.56        86
        Good       0.71      0.83      0.76       148
     Neutral       0.00      0.00      0.00        23

    accuracy                           0.66       257
   macro avg       0.42      0.46      0.44       257
weighted avg       0.60      0.66      0.63       257



### 2.3.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** Justifica la elección de tus modelos base y meta-modelo: ¿por qué los elegiste y qué aporta cada uno? ¿El Stacking mejoró respecto a los modelos base individuales de las secciones anteriores? ¿A qué atribuyes ese resultado considerando cómo funciona el algoritmo?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

# 3. Optimización de Hiperparámetros con Optuna [1.5 Puntos]

<p align="center">
  <img src="https://media.giphy.com/media/3oKIPEqDGUULpEU0aQ/giphy.gif" width="300">
</p>

Hasta ahora hemos usado hiperparámetros por defecto. En esta sección optimizaremos **LightGBM** con **Optuna**, una librería de optimización bayesiana más eficiente que `GridSearchCV` porque:

- **Samplers inteligentes (TPE)**: usa el historial de trials para proponer mejores valores.
- **Pruning**: abandona trials poco prometedores antes de completarlos.
- **Visualizaciones interactivas**: analiza el espacio de hiperparámetros con `optuna-dashboard`.

Los resultados del estudio se persistirán en una base de datos SQLite, lo que permite explorarlos en tiempo real con el dashboard.

### 3.1 Descripción: Hiperparámetros y Optimización [0.2 Puntos]

Antes de implementar la optimización, reflexiona sobre los siguientes conceptos:

1. **¿Qué son los hiperparámetros?** ¿En qué se diferencian de los parámetros que el modelo aprende durante el entrenamiento?
2. **¿Por qué es importante optimizarlos?** ¿Qué consecuencias puede tener usar hiperparámetros por defecto?
3. **¿Qué es `GridSearchCV`?** Describe cómo funciona su mecanismo de búsqueda y cuál es su principal limitación cuando el espacio de hiperparámetros es grande.
4. **¿Por qué se evalúa cada trial con validación cruzada en lugar de usar directamente el conjunto de test?** ¿Qué problema introduce usar el test para seleccionar hiperparámetros? ¿Cómo podríamos reemplazar la CV por un conjunto de validación separado, y cuáles serían las ventajas y desventajas de ese enfoque?
5. **¿Cómo mejoran los samplers inteligentes la limitación de `GridSearchCV`?** ¿Por qué un sampler como TPE puede ser más eficiente que una búsqueda exhaustiva?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

### 3.2 Definición del Espacio de Búsqueda y Función Objetivo [0.5 Puntos]

La función `objective(trial)` recibe un trial de Optuna, define los hiperparámetros a probar y devuelve la métrica a optimizar (F1-Macro).

| Hiperparámetro | Tipo | Rango | Descripción |
|----------------|------|-------|-------------|
| `num_leaves` | `suggest_int` | [20, 200] | Número máximo de hojas por árbol; controla la complejidad del modelo. |
| `learning_rate` | `suggest_float` (log) | [1e-3, 0.3] | Tasa de aprendizaje; cuánto contribuye cada árbol a la predicción final. |
| `max_depth` | `suggest_int` | [3, 12] | Profundidad máxima de cada árbol; limita la capacidad de memorización. |
| `min_child_samples` | `suggest_int` | [5, 100] | Mínimo de muestras requeridas en una hoja; regulariza contra sobreajuste. |
| `subsample` | `suggest_float` | [0.5, 1.0] | Fracción de filas muestreadas aleatoriamente para entrenar cada árbol. |
| `colsample_bytree` | `suggest_float` | [0.5, 1.0] | Fracción de features muestreadas aleatoriamente para cada árbol. |
| `reg_alpha` | `suggest_float` (log) | [1e-8, 10.0] | Regularización L1 sobre los pesos de las hojas. |
| `reg_lambda` | `suggest_float` (log) | [1e-8, 10.0] | Regularización L2 sobre los pesos de las hojas. |

**To-do:**
- [ ] Implementar `objective(trial)` con el espacio de búsqueda indicado.
- [ ] Usar `StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)`.
- [ ] Retornar la media del F1-Macro sobre los 3 folds.

In [ ]:
#### Código aquí ####


In [17]:
def objective(trial):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 20, 200),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }

    pipe = Pipeline(
        [
            ("features", preprocessing_transformer),
            ("clf", LGBMClassifier(**params, random_state=RANDOM_STATE, verbose=-1)),
        ]
    )

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1_macro")
    return scores.mean()

### 3.3 Ejecución del Estudio de Optimización [0.4 Puntos]

Crea el estudio Optuna con 50 trials usando `TPESampler`. El estudio se guarda en el storage SQLite configurado anteriormente para poder explorarlo con `optuna-dashboard`.

**To-do:**
- [ ] Crear el estudio con `direction="maximize"`, `TPESampler(seed=RANDOM_STATE)` y el `storage` configurado.
- [ ] Ejecutar `study.optimize` con `n_trials=50` y `show_progress_bar=True`
- [ ] Imprimir el mejor valor y los mejores hiperparámetros.

In [18]:
storage = optuna.storages.RDBStorage("sqlite:///optuna_lab7.db")
study = optuna.create_study(
    direction="maximize",
    study_name="lgbm-comics",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    storage=storage,
    load_if_exists=True,
)

study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Mejor F1-Macro: {study.best_value:.4f}")
print("\nMejores hiperparámetros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

Best trial: 21. Best value: 0.457204: 100%|██████████| 50/50 [40:10<00:00, 48.20s/it]  

Mejor F1-Macro: 0.4572

Mejores hiperparámetros:
  num_leaves: 60
  learning_rate: 0.28754519962429337
  max_depth: 7
  min_child_samples: 23
  subsample: 0.8456742564627487
  colsample_bytree: 0.8941468837236076
  reg_alpha: 7.983960331592793e-05
  reg_lambda: 0.12446945119532377


### 3.4 Visualizaciones de Optuna [0.2 Puntos]

Optuna provee visualizaciones interactivas para analizar el proceso de optimización.

**To-do:**
- [ ] Graficar el historial de optimización (`plot_optimization_history`).
- [ ] Graficar la importancia de hiperparámetros (`plot_param_importances`).
- [ ] Graficar una visualización del comportamiento de los hiperparámetros (`plot_parallel_coordinate`)

In [23]:
#### Código aquí ####
#Grafico del historial de optimización
fig_history = plot_optimization_history(study)
fig_history.show() #Este gráfico muestra como evoluciona el valor objetivo F1-Macro a lo largo de los trials.

# Grafico la importancia de los hiperparámetros
fig_importance = plot_param_importances(study)
fig_importance.show() #En este caso, el hiperparametro que más afecto al F1-macro fue el learning_rate, seguido por min_child_sample y max_depth. Esto sugiere que ajustar estos hiperparámetros podría tener un impacto significativo en el rendimiento del modelo.

#Grafico ddel comportamiento de los hiperparámetros
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show() #Este gráfico muestra como los diferentes hiperparámetros interactúan entre sí

### 3.5 Comparar Modelo Base con Mejores Hiperparámetros [0.1 Puntos]

**To-do:**
- [ ] Reentrenar un pipeline `pipe_lgbm_opt` con `study.best_params`.
- [ ] Comparar F1-Macro entre `pipe_lgbm` (defaults) y `pipe_lgbm_opt` (optimizado).

In [24]:
#### Código aquí ####
pipeline_lgbm_opt = Pipeline(
    steps=[
        ("preprocesamiento", preprocessing_transformer),
        ("modelo", LGBMClassifier(**study.best_params, random_state=RANDOM_STATE, verbose=-1)) #Verbose = -1 para evitar mensajes de LightGBM durante el entrenamiento
    ]
)

pipeline_lgbm_opt.fit(X_train, y_train)
y_pred_lgbm_opt = pipeline_lgbm_opt.predict(X_test)

print("Classification Report - LightGBM Optimizado:")
print(classification_report(y_test, y_pred_lgbm_opt))

Classification Report - LightGBM Optimizado:
              precision    recall  f1-score   support

         Bad       0.54      0.50      0.52        86
        Good       0.71      0.82      0.76       148
     Neutral       0.67      0.17      0.28        23

    accuracy                           0.66       257
   macro avg       0.64      0.50      0.52       257
weighted avg       0.65      0.66      0.64       257



In [25]:
print("Comparativa de F1-Macro de ambos modelos LightGBM:")
f1_lgbm_default = f1_score(y_test, y_pred_LGBMC, average="macro")
f1_lgbm_opt = f1_score(y_test, y_pred_lgbm_opt, average="macro")
print(f"LightGBM por defecto: {f1_lgbm_default:.4f}")
print(f"LightGBM optimizado: {f1_lgbm_opt:.4f}")

Comparativa de F1-Macro de ambos modelos LightGBM:
LightGBM por defecto: 0.4533
LightGBM optimizado: 0.5199


### 3.6 Preguntas de reflexión [0.1 Puntos]

1. Según `plot_param_importances`, ¿qué hiperparámetro tuvo más impacto? ¿Tiene sentido dado tu conocimiento sobre algoritmos de Boosting?
2. ¿Qué información adicional te entregaron los gráficos de optuna comparado con solo mirar los logs de texto del estudio? ¿Qué pasa si en el gráfico interactivo `plot_parallel_coordinate` seleccionas con el mouse el rango específico con mejores resultados, hay patrones interesantes? - Hint: hace drag con el puntero sobre la dimensión de _Objective Value_ para seleccionar las mejores observaciones.

In [ ]:
# Escribe aquí la respuesta

# 4. Interpretabilidad [1 Punto]

<p align="center">
  <img src="https://media.giphy.com/media/xT9IgzoKnwFNmISR8I/giphy.gif" width="300">
</p>

En esta sección interpretaremos las predicciones de LightGBM. Para obtener interpretaciones claras y directas, entrenaremos un **modelo auxiliar de LGBM usando únicamente los 6 atributos numéricos** (scores de habilidades), sin incluir features de texto.

Esto nos permite:
- Aplicar PFI y SHAP sin complicaciones de alta dimensionalidad.
- Obtener explicaciones semánticamente significativas (fuerza, inteligencia, velocidad...).

> **Nota:** En esta sección entrenaremos un modelo auxiliar para interpretabilidad con menos atributos. Usaremos un modelo entrenado desde 0, no el mejor seleccionado anteriormente.

In [ ]:
NUMERICAL_FEATURES = [
    "intelligence_score",
    "strength_score",
    "speed_score",
    "durability_score",
    "power_score",
    "combat_score",
]

X_train_num = X_train[NUMERICAL_FEATURES]
X_test_num = X_test[NUMERICAL_FEATURES]

# Reentrenar LGBM con los mejores hiperparámetros, solo sobre scores numéricos
lgbm_interp = ...

## 4.1 Permutation Feature Importance (PFI) [0.4 Puntos]

### 4.1.1 Descripción [0.1 Puntos]

Responde las siguientes preguntas con tus propias palabras:

1. **¿Cómo funciona la Permutation Feature Importance?** ¿Qué se permuta exactamente y cómo se mide el impacto en el rendimiento del modelo?
2. **¿Por qué PFI es preferible a la importancia nativa de los árboles de decisión?** ¿Qué sesgo tiene la importancia nativa que PFI evita?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

### 4.1.2 Implementación [0.2 Puntos]

**To-do:**
- [ ] Calcular `permutation_importance` con `n_repeats=30` y `scoring="f1_macro"`.
- [ ] Graficar un boxplot horizontal con plotly (`px.bar` con `orientation='h'`) con la importancia y varianza de cada feature.

In [ ]:
#### Código aquí ####

### 4.1.3 Pregunta de Cierre [0.1 Puntos]

**Pregunta:** Según el gráfico de PFI, ¿qué feature tuvo mayor importancia? ¿Tiene sentido intuitivo dado el contexto del problema (clasificar la alineación de personajes de cómics)? ¿Hay alguna feature cuya importancia te sorprenda?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

## 4.2 SHAP (SHapley Additive exPlanations) [0.6 Puntos]

### 4.2.1 Descripción [0.2 Puntos]

Responde las siguientes preguntas con tus propias palabras:

1. **¿Qué miden los SHAP values?** ¿En qué se diferencian de una medida de importancia global como PFI?
2. **¿Qué representa el `base_value` en un `waterfall_plot` de SHAP?** ¿Por qué es el punto de partida para interpretar una predicción individual?
3. **¿Por qué usar `TreeExplainer` para modelos basados en árboles?** ¿Qué ventaja ofrece respecto a un explainer genérico?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

### 4.2.2 Implementación [0.2 Puntos]

**To-do:**
- [ ] Crear un `shap.TreeExplainer(lgbm_interp)` y calcular `shap_values` sobre `X_test_num`.
- [ ] `summary_plot` para ver importancia global y dirección del efecto (para la clase "Good").
- [ ] `waterfall_plot` para la predicción de una instancia cualquiera de `X_test_num`.

In [ ]:
#### Código aquí ####

### 4.2.3 Pregunta de Cierre [0.2 Puntos]

1. ¿Qué diferencia existe entre Permutation Feature Importance y los SHAP values como medida de importancia de features?
2. Según el `waterfall_plot`, ¿qué features fueron las que más empujaron la predicción hacia su clase? Investiga el personaje seleccionado: ¿Tiene sentido dado su historia en los cómics?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

---

# 5. Predicción de Personajes No Etiquetados [0.5 Puntos]

<p align="center">
  <img src="https://pbs.twimg.com/media/DolotxUUYAAbg7f.jpg" width="350">
</p>

¡Llegó el momento de predecir `Vergil`, `Gorilla Girl` y `Bat-Cow`!

Usaremos el **mejor modelo** obtenido en la sección 3 (`pipe_lgbm_opt`) para predecir la alineación de los personajes no etiquetados.

**Nota:** Recuerda eliminar los NaN en `history_text` antes de predecir.

### 5.0 Predicción [0.2 Puntos]

**To-do:**
- [ ] Usar `pipe_lgbm_opt` para predecir `alignment` en `df_comics_no_label` (recuerda eliminar NaN en `history_text`).
- [ ] Filtrar y mostrar resultados para `Vergil`, `Gorilla Girl` y `Bat-Cow`.

In [ ]:
#### Código aquí ####

### 5.1 Análisis de Predicciones [0.3 Puntos]

**Pregunta:** Comenta las predicciones obtenidas para `Vergil`, `Gorilla Girl` y `Bat-Cow`:

1. ¿Las predicciones te parecen razonables según lo que conoces (o puedes inferir) de estos personajes?
2. Conecta con la sección 4: ¿qué features numéricas habrían influido más en la predicción de **Bat-Cow** según el `waterfall_plot`? ¿Es consistente con la predicción obtenida aquí?

> **Respuesta:**

In [ ]:
# Escribe aquí la respuesta

# Conclusión

¡Eso ha sido todo para el lab de hoy! Recuerden que el laboratorio tiene un plazo de entrega de una semana y que **los días de atraso no se pueden utilizar para entregas de lab, solo para tareas**. Cualquier duda del laboratorio, no duden en contactarnos por mail o U-cursos.

<p align="center">
  <img src="https://media1.tenor.com/images/fb5bf7cc5a4acb91b4177672886a88ba/tenor.gif?itemid=5591338">
</p>